In [ ]:
#This file is based on FootNet's implementation of identifying a strikefoot. 
#A few limitations to address are that FootNet's research lab had 33 keypoints, obtained from motion captrue data whereas YOLOv8's pose models only offer 17 (half that)
#Another limitation lies in the fact that the lab had forceplates to recognise when strikefoot's took place (on training data). I will have to comb through the frames and manually see where strikefoot takes place
import cv2
from collections import defaultdict
from video_extracter import extract_keypoints
import numpy as np

final = defaultdict()
vid_num = 4

#This function will go through the video frame by frame. Each frame as classified as "u" = left foot up, "d" = left foot down, "s" = skip (if its neither).
#"u" and "d" apply to the EXACT moment the foot goes up or down respectively. Each consecutive pair of "u" and "d" will be classified as a gait cycle
def extract_gait_cycle(video_dir):
    gait_cycles = []
    single_gait_cycle = []
    cap = cv2.VideoCapture(video_dir)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    for i in range(frame_count):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imshow('Frame', frame)
        key = cv2.waitKey(0)
        #Constructing a single gait cycle and appending them
        if key == ord('u'):
            single_gait_cycle.append(i)
        elif key == ord('d'):
            single_gait_cycle.append(i)
            gait_cycles.append(single_gait_cycle)
            single_gait_cycle = []
        elif key == ord('c'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    return gait_cycles

#The purpose of this function is to extract the following 4 metrics
#1) Velocity of the shin (Velocity of the x component of the vector between the ankle and the knee)
#2) Tibial angle
#3) Velocity of the ankle of the x component
#4) Velocity of the ankle of the y component
def extract_metrics(video_dir):
    coords = extract_keypoints(video_dir)
    right_ankle_coords = coords["right_ankle"]
    left_ankle_coords = coords["left_ankle"]
    left_knee_coords = coords["left_knee"]
    right_knee_cords = coords["right_knee"]
    frame_count = coords["Frame Count"]

    right_ankle_y_vel = np.gradient(right_ankle_coords[:,1])
    right_ankle_x_vel = np.gradient(right_ankle_coords[:,0])
    left_ankle_x_vel = np.gradient(left_ankle_coords[:,0])
    left_ankle_y_vel = np.gradient(left_ankle_coords[:,1])
    shin_vector_right = right_knee_cords - right_ankle_coords
    shin_velocity_right = np.gradient(shin_vector_right[:, 0])
    shin_vector_left = left_knee_coords - left_ankle_coords
    shin_velocity_left = np.gradient(shin_vector_left[:, 0])

    def obtain_tibial_angles(shin_vectors):
        angles = []
        for vector in shin_vectors:
            angle = np.arctan2(vector[1], vector[0])
            angle = np.pi/2 - angle
            angles.append(angle)

        angles = np.array(angles)
        return angles
    left_tibial = obtain_tibial_angles(shin_vector_left)
    right_tibial = obtain_tibial_angles(shin_vector_right)

    return {
    "right_ankle_x_vel": right_ankle_x_vel,
    "right_ankle_y_vel": right_ankle_y_vel,
    "left_ankle_x_vel": left_ankle_x_vel,
    "left_ankle_y_vel": left_ankle_y_vel,
    "shin_velocity_left": shin_velocity_left,
    "shin_velocity_right": shin_velocity_right,
    "left_tibial": left_tibial,
    "right_tibial": right_tibial,
    "frame count": frame_count
    }

#Initial jsondict that will be built from the ground up
json_dict = {}

In [ ]:
def construct_json(final:dict, video_dir = "/Users/abhinavarora/Desktop/CadenceCV/Videos/Normal Running.MOV", video_num = 1):
    #These metrics must be extracted for ALL frames in each video
    metrics = extract_metrics(video_dir)
    frame_count = metrics["frame count"]
    #Manual labelling done via tracking the left foot
    left_foot_cycle = extract_gait_cycle(video_dir)
    #Ths time, the right foot will be focused on to identify its gait cycles.
    right_foot_cycle = extract_gait_cycle(video_dir)
    strikefoot_frames = set()

    intermediate_dict = {}
    #"d" is always the strikefoot frame
    for arr in left_foot_cycle:
        strikefoot_frames.add((arr[1], "L"))
    
    for arr in right_foot_cycle:
        strikefoot_frames.add((arr[1], "R"))

    #Obtaining metrics for each individual frame
    for frame in range(frame_count):
        frame_metrics = []
        for m in metrics:
            if metrics[m] is isinstance(list[np.floating]):
                #Checking in case YOLO couldn't pick up on certain metrics, meaning it went out of bounds
                frame_metrics.append(metrics[m][frame]) if frame < len(metrics) else frame_metrics.append(None)

        if (frame, "L") in strikefoot_frames:
            frame_metrics.append("L")
            intermediate_dict[frame] = frame_metrics
        elif (frame, "R") in strikefoot_frames:
            frame_metrics.append("R")
            intermediate_dict[frame] = frame_metrics
    
    final["Video" + video_num] = intermediate_dict

In [ ]:
import torch
import pandas as pd
from pandas import json_normalize
import json

#Extracting features from the JSON file first
with open("strikefoot_recogniser_data.json", "r") as file:
    data = json.load(file)

dataframes = []
for i in range(vid_num):
    df = json_normalize(data["Video" + i])
    dataframes.append(df)

#Joining all dataframes into 1 dataframe
df = pd.concat(dataframes, axis=0, ignore_index=True)